In [ ]:
import os
import scanpy as sc
import numpy as np

import scvi
from scvi.external import RNAStereoscope, SpatialStereoscope

celltype_key = "celltype_final"

for dataset_number in range(1, 33):
    sc_file = f"/home/maweicheng/ST_data/SimualtedSpatalData/dataset{dataset_number}/scRNA.h5ad"
    st_file = f"/home/maweicheng/ST_data/SimualtedSpatalData/dataset{dataset_number}/Spatial.h5ad"
    output_dir = f"/home/maweicheng/ST_data/SimualtedSpatalData/evaluate/Data{dataset_number}"
    os.makedirs(output_dir, exist_ok=True)

    print(f"=== Data {dataset_number} ===")
    print(">>> Loading data")
    sc_adata = sc.read_h5ad(sc_file)
    st_adata = sc.read_h5ad(st_file)

    # ----------------- 1. 单细胞预处理（Stereoscope 版本） -----------------
    print(">>> Preprocessing scRNA")

    # 过滤掉低表达基因
    sc.pp.filter_genes(sc_adata, min_counts=10)

    # 去掉线粒体基因（MT- 开头），保持和你之前脚本一致
    non_mito_genes = [g for g in sc_adata.var_names if not g.startswith("MT-")]
    sc_adata = sc_adata[:, non_mito_genes]

    # 原始 counts 存在 layer，给 Stereoscope 用
    sc_adata.layers["counts"] = sc_adata.X.copy()

    # 归一化 + log1p（主要给可视化 / 其它分析用）
    sc.pp.normalize_total(sc_adata, target_sum=1e5)
    sc.pp.log1p(sc_adata)
    sc_adata.raw = sc_adata

    # HVG（在 counts 上选 7000 个）
    sc.pp.highly_variable_genes(
        sc_adata,
        n_top_genes=7000,
        subset=True,
        layer="counts",
        flavor="seurat_v3",
        span=1,
    )

    # ----------------- 2. 空间数据预处理 -----------------
    print(">>> Preprocessing spatial")

    # 原始 counts 存在 layer，给 Stereoscope 用
    st_adata.layers["counts"] = st_adata.X.copy()

    # 归一化 + log1p
    sc.pp.normalize_total(st_adata, target_sum=1e5)
    sc.pp.log1p(st_adata)
    st_adata.raw = st_adata

    # 基因交集（保证两个模型看到的基因一致）
    intersect = np.intersect1d(sc_adata.var_names, st_adata.var_names)
    sc_adata = sc_adata[:, intersect].copy()
    st_adata = st_adata[:, intersect].copy()
    print(f">>> Shared genes: {len(intersect)}")

    # ----------------- 3. 训练 RNAStereoscope（sc 侧） -----------------
    print(">>> Training RNAStereoscope")

    RNAStereoscope.setup_anndata(
        sc_adata,
        layer="counts",
        labels_key=celltype_key,
    )

    stereo_sc_model = RNAStereoscope(sc_adata)
    stereo_sc_model.train(max_epochs=100)

    # ----------------- 4. 训练 SpatialStereoscope（ST 侧） -----------------
    print(">>> Training SpatialStereoscope")

    SpatialStereoscope.setup_anndata(
        st_adata,
        layer="counts",
    )

    spatial_model = SpatialStereoscope.from_rna_model(st_adata, stereo_sc_model)
    spatial_model.train(max_epochs=10000)

    # ----------------- 5. 导出 proportion -----------------
    print(">>> Saving results")
    proportions = spatial_model.get_proportions()
    proportions.to_csv(os.path.join(output_dir, "Stereoscope.csv"))

    print(f">>> Data {dataset_number} done.\n")


In [1]:
import os
import scanpy as sc
import numpy as np

import scvi
from scvi.external import RNAStereoscope, SpatialStereoscope


celltype_key = "celltype"
sc_file = f"/home/maweicheng/ST_data/STARmap/starmap_sc_rna.h5ad"
st_file = f"/home/maweicheng/ST_data/STARmap/STARmap_SP.h5ad"
output_dir = f"/home/maweicheng/ST_data/STARmap"
os.makedirs(output_dir, exist_ok=True)

print(">>> Loading data")
sc_adata = sc.read_h5ad(sc_file)
st_adata = sc.read_h5ad(st_file)

# ----------------- 1. 单细胞预处理（Stereoscope 版本） -----------------
print(">>> Preprocessing scRNA")

# 过滤掉低表达基因
sc.pp.filter_genes(sc_adata, min_counts=10)

# 去掉线粒体基因（MT- 开头），保持和你之前脚本一致
non_mito_genes = [g for g in sc_adata.var_names if not g.startswith("MT-")]
sc_adata = sc_adata[:, non_mito_genes]

# 原始 counts 存在 layer，给 Stereoscope 用
sc_adata.layers["counts"] = sc_adata.X.copy()

# 归一化 + log1p（主要给可视化 / 其它分析用）
sc.pp.normalize_total(sc_adata, target_sum=1e5)
sc.pp.log1p(sc_adata)
sc_adata.raw = sc_adata

# HVG（在 counts 上选 7000 个）
sc.pp.highly_variable_genes(
    sc_adata,
    n_top_genes=7000,
    subset=True,
    layer="counts",
    flavor="seurat_v3",
    span=1,
)

# ----------------- 2. 空间数据预处理 -----------------
print(">>> Preprocessing spatial")

# 原始 counts 存在 layer，给 Stereoscope 用
st_adata.layers["counts"] = st_adata.X.copy()

# 归一化 + log1p
sc.pp.normalize_total(st_adata, target_sum=1e5)
sc.pp.log1p(st_adata)
st_adata.raw = st_adata

# 基因交集（保证两个模型看到的基因一致）
intersect = np.intersect1d(sc_adata.var_names, st_adata.var_names)
sc_adata = sc_adata[:, intersect].copy()
st_adata = st_adata[:, intersect].copy()
print(f">>> Shared genes: {len(intersect)}")

# ----------------- 3. 训练 RNAStereoscope（sc 侧） -----------------
print(">>> Training RNAStereoscope")

RNAStereoscope.setup_anndata(
    sc_adata,
    layer="counts",
    labels_key=celltype_key,
)

stereo_sc_model = RNAStereoscope(sc_adata)
stereo_sc_model.train(max_epochs=100)

# ----------------- 4. 训练 SpatialStereoscope（ST 侧） -----------------
print(">>> Training SpatialStereoscope")

SpatialStereoscope.setup_anndata(
    st_adata,
    layer="counts",
)

spatial_model = SpatialStereoscope.from_rna_model(st_adata, stereo_sc_model)
spatial_model.train(max_epochs=10000)

# ----------------- 5. 导出 proportion -----------------
print(">>> Saving results")
proportions = spatial_model.get_proportions()
proportions.to_csv(os.path.join(output_dir, "Stereoscope.csv"))

print(f">>> Data {dataset_number} done.\n")


/softwares/miniconda3/envs/cell2loc_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


>>> Loading data
>>> Preprocessing scRNA


/tmp/ipykernel_2637295/4225142922.py:30: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  sc_adata.layers["counts"] = sc_adata.X.copy()


>>> Preprocessing spatial


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


...

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/softwares/miniconda3/envs/cell2loc_env/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=111` in the `DataLoader` to improve performance.
/softwares/miniconda3/envs/cell2loc_env/lib/python3.9/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (2) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for log_every_n_steps if you want t


>>> Training SpatialStereoscope
Epoch 10000/10000: 100%|██████████| 10000/10000 [03:48<00:00, 41.21it/s, v_num=1, train_loss_step=1.39e+5, train_loss_epoch=1.42e+5]

`Trainer.fit` stopped: `max_epochs=10000` reached.


Epoch 10000/10000: 100%|██████████| 10000/10000 [03:48<00:00, 43.74it/s, v_num=1, train_loss_step=1.39e+5, train_loss_epoch=1.42e+5]
>>> Saving results


NameError: name 'dataset_number' is not defined